# Panerai ETL Workflow Test
This notebook implements the incremental ETL pipeline from Bronze $\rightarrow$ Silver $\rightarrow$ Gold layers.

In [ ]:
import os
from pyspark.sql import functions as F
from pyspark.sql import Window
import yaml

# --- Configuration ---
ENV = "dev"
BRONZE_TABLE = f"panerai_project.{ENV}.panerai_data_bronze_{ENV}"
SILVER_TABLE = f"panerai_project.{ENV}.panerai_data_silver_{ENV}"
VOLUME_PATH = f"/Volumes/panerai_project/{ENV}/panerai_data"
EXCHANGE_RATES = {
    "USD": 0.88, "JPY": 0.0056, "AED": 0.24, 
    "CHF": 1.06, "GBP": 1.16, "EUR": 1.0
}

## 1. Bronze Layer: Incremental Ingestion
Using Auto Loader to ingest new files from the Volume.

In [ ]:
# In a real production job, we would use spark.readStream with cloudFiles
# For the test notebook, we simulate incremental load by reading the volume
bronze_df = spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(VOLUME_PATH)

# Adding metadata for traceability (simulating _metadata.file_path)
bronze_df = bronze_df.withColumn("file_path", F.input_file_name())

display(bronze_df.limit(5))

## 2. Silver Layer: Cleaning & Normalization
Processing the raw data and preparing for a MERGE into the Silver table.

In [ ]:
def clean_silver_data(df):
    # 1. Drop rows without reference
    df = df.dropna(subset=["reference"])
    
    # 2. Clean Numeric Columns
    df = df.withColumn("price_clean", F.regexp_replace(F.col("price"), r"[^0-9]", "").cast("int"))
    df = df.withColumn("diameter_mm", F.regexp_replace(F.col("diameter_tag"), r"[^0-9]", "").cast("int"))
    
    # 3. Currency Conversion
    # Create a mapping column based on the EXCHANGE_RATES dict
    rate_expr = F.lit(1.0)
    for curr, rate in EXCHANGE_RATES.items():
        rate_expr = F.when(F.col("currency") == curr, F.lit(rate)).otherwise(rate_expr)
    
    df = df.withColumn("price_eur", F.col("price_clean") * rate_expr)
    
    # 4. Missing Value Handling
    df = df.withColumn("edition", F.coalesce(F.col("edition"), F.lit("unknown")))
    df = df.withColumn("comment", F.coalesce(F.col("comment"), F.lit("unknown")))
    
    # 5. Price Display Logic
    df = df.withColumn("price_display", 
                      F.when(F.col("price_clean").isNull(), F.lit("Contact store directly")).otherwise(F.col("price_clean").cast("string")))
    
    return df

silver_processed_df = clean_silver_data(bronze_df)
display(silver_processed_df.limit(5))

## 3. Silver Layer: Incremental Merge
Instead of Overwriting, we Merge into the target table.

In [ ]:
# First time setup: Create table if not exists
spark.sql(f"CREATE TABLE IF NOT EXISTS {SILVER_TABLE} AS SELECT * FROM (SELECT 1 as dummy) WHERE 1=0")

# To perform a MERGE, we need a temporary view
silver_processed_df.createOrReplaceTempView("updates")

spark.sql(f"""
MERGE INTO {SILVER_TABLE} AS target
USING updates AS source
ON target.reference = source.reference 
   AND target.country = source.country 
   AND target.date = source.date
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

print("Silver layer merged successfully.")

## 4. Gold Layer: Star Schema
Transforming the flat Silver table into Dimensions and Facts.

In [ ]:
silver_final = spark.read.table(SILVER_TABLE)

# Dimension: Collection
dim_collection = silver_final.select("collection").distinct().withColumn("collection_id", F.monotonically_increasing_id())

# Dimension: Country
dim_country = silver_final.select("country", "currency").distinct().withColumn("country_id", F.monotonically_increasing_id())

# Dimension: Watch
dim_watch = silver_final.select("reference", "title", "edition", "comment").distinct()

# Fact Table
fact_pricing = silver_final.join(dim_collection, "collection") \
                           .join(dim_country, "country") \
                           .select(
                               "reference", 
                               "date", 
                               "price_eur", 
                               "diameter_mm", 
                               "collection_id", 
                               "country_id"
                           )

display(fact_pricing.limit(5))